In [55]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Image, display

In [56]:
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

In [57]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation : str

In [58]:
def joke_generator(state : JokeState):
    """ Joke generator node """
    messages = [SystemMessage(content="You are a professional joke generator, your task is to generate funny jokes"),
                HumanMessage(content=f"Generate a joke on this topic : {state["topic"]}")]
    resposne = llm.invoke(messages).content[-1]["text"]
    return {"joke" : resposne}

In [59]:
def explanation_generator(state : JokeState):
    """ Joke's Explanation generator node """
    messages = [SystemMessage(content="You are a professional explanation generator assistant, your task is to generate an explanation of the joke on a given toic. The explanation should not be more than 250 words and should be easy to understand."),
                HumanMessage(content=f"For the topic '{state["topic"]}', generate an explanation for the joke : {state["joke"]} ")]
    response  = llm.invoke(messages).content[-1]["text"]
    return {"explanation" : response }


In [60]:
graph = StateGraph(JokeState)
graph.add_node("joke_generator", joke_generator)
graph.add_node("explanation_generator", explanation_generator)


In [61]:
graph.add_edge(START, "joke_generator")
graph.add_edge("joke_generator", "explanation_generator")
graph.add_edge("explanation_generator", END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [62]:
config1 = {"configurable" : {"thread_id" : 1}}

In [63]:
initial_state = {
    "topic" : "Muslim"
}
workflow.invoke(initial_state, config=config1)

{'topic': 'Muslim',
 'joke': 'A young boy was fasting for the first time during Ramadan. By mid-afternoon, he was feeling pretty exhausted and hungry.\n\nHe went up to his father and said, "Dad, I’m so hungry, I feel like I’m going to collapse!"\n\nHis father, trying to encourage him, said, "Stay strong, son! Just think about the reward. In Jannah (Paradise), you can have anything you want to eat, anytime you want it!"\n\nThe boy looked up with big, tired eyes and asked, "Can we just go there for lunch and come back?"',
 'explanation': 'This joke relies on the cultural context of **Ramadan**, the Islamic holy month during which Muslims fast from dawn until sunset. The humor stems from the innocent, literal perspective of a child struggling with the physical demands of fasting.\n\nThere are three key elements that make this joke work:\n\n1.  **The Concept of Jannah:** In Islam, *Jannah* (Paradise) is described as the ultimate reward for faith and patience. It is believed to be a place w

In [64]:
workflow.get_state(config1)
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Muslim', 'joke': 'A young boy was fasting for the first time during Ramadan. By mid-afternoon, he was feeling pretty exhausted and hungry.\n\nHe went up to his father and said, "Dad, I’m so hungry, I feel like I’m going to collapse!"\n\nHis father, trying to encourage him, said, "Stay strong, son! Just think about the reward. In Jannah (Paradise), you can have anything you want to eat, anytime you want it!"\n\nThe boy looked up with big, tired eyes and asked, "Can we just go there for lunch and come back?"', 'explanation': 'This joke relies on the cultural context of **Ramadan**, the Islamic holy month during which Muslims fast from dawn until sunset. The humor stems from the innocent, literal perspective of a child struggling with the physical demands of fasting.\n\nThere are three key elements that make this joke work:\n\n1.  **The Concept of Jannah:** In Islam, *Jannah* (Paradise) is described as the ultimate reward for faith and patience. It is beli

In [65]:
config2 = {"configurable" : {"thread_id" : 2}}
workflow.invoke({"topic" : "Kutta"}, config=config2)


{'topic': 'Kutta',
 'joke': 'Here is a funny joke about a dog (Kutta) in Hinglish:\n\n**Pappu apne dost Golu se:** "Bhai, mera kutta ekdum \'Police Dog\' hai!"\n\n**Golu:** "Lekin ye toh kahin se bhi Police Dog nahi lag raha, ye toh ekdum dubla-patla aur sadak-chhap lag raha hai."\n\n**Pappu:** "Arre bhai, dhyan se dekh... ye **Undercover** mission pe hai!"\n\n***\n\n**Bonus Joke:**\n\n**Pappu:** "Bhai, tera kutta toh bilkul insaano jaisa lagta hai."\n\n**Golu:** "Haan bhai, lekin thoda badtameez hai."\n\n**Pappu:** "Woh kaise?"\n\n**Golu:** "Jab bhi main isey bolta hoon \'Tommy, haath milao\', toh ye mujhe dekh kar kehta hai—**\'Pehle biscuit nikaal, rishtedaari baad mein nibhayenge!\'**"',
 'explanation': 'These jokes use clever wordplay and human-like traits to create humor around the topic of a dog (**Kutta**).\n\n**Explanation of Joke 1: The Undercover Dog**\nThe humor in the first joke comes from a witty justification. Usually, "Police Dogs" are expected to be strong, well-bred, 

Time Travel

In [ ]:

workflow.get_state({"configurable" : {"thread_id" : 1, "checkpoint_id" : '1f1eafe6-d722-6438-8000-34fe4f27d3d1'}})

StateSnapshot(values={'topic': 'Muslim'}, next=('joke_generator',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1eafe6-d722-6438-8000-34fe4f27d3d1'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-11-28T07:36:30.398563+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1eafe6-d71c-665a-bfff-29adcde96a5d'}}, tasks=(PregelTask(id='619f3d2d-de0c-5a91-74bd-27dfb6ea3181', name='joke_generator', path=('__pregel_pull', 'joke_generator'), error=None, interrupts=(), state=None, result={'joke': 'A young boy was fasting for the first time during Ramadan. By mid-afternoon, he was feeling pretty exhausted and hungry.\n\nHe went up to his father and said, "Dad, I’m so hungry, I feel like I’m going to collapse!"\n\nHis father, trying to encourage him, said, "Stay strong, son! Just think about the reward. In Jannah (Paradise), you can have anything you want to eat, anytime you want it!"\n\nThe boy looked up wi

In [69]:
workflow.invoke(None, {"configurable" : {"thread_id" : 1, "checkpoint_id" : '1f1eafe6-d722-6438-8000-34fe4f27d3d1'}} )

{'topic': 'Muslim',
 'joke': 'A man is walking down the street when he sees his friend, Omar, looking very stressed.\n\n"Omar, what’s wrong?" he asks.\n\nOmar sighs, "I\'m trying to teach my five-year-old son how to pray, but it’s not going well."\n\n"Why? Is he having trouble with the movements?" his friend asks.\n\n"No," Omar replies. "But every time I stand up and say *\'Allahu Akbar\'* to start the prayer, he looks at me and whispers, \'Dad, why are you calling Admiral Ackbar from *Star Wars*? Is it a trap?\'"',
 'explanation': 'This joke relies on a humorous phonetic misunderstanding between a sacred religious phrase and a famous pop culture reference.\n\nIn the Islamic faith, when a person begins their formal prayer (Salah), they raise their hands and say **"Allahu Akbar,"** which is Arabic for "God is the Greatest." This phrase, known as the *Takbir*, is the standard way to signal the start of the prayer.\n\nThe humor comes from Omar’s five-year-old son, who confuses the Arabic 

In [80]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?\n\nBecause it was **all crust on the outside, but totally mashed on the inside!**', 'explanation': 'This joke is a clever play on words that uses the physical characteristics of a popular snack—the **samosa**—as a metaphor for human emotions.\n\nTo understand the humor, you first have to look at the literal structure of a samosa. It consists of a thick, deep-fried pastry shell that is firm and crunchy, commonly referred to as the **crust**. Inside that shell, the filling is almost always made of soft, spiced potatoes that have been completely **mashed**.\n\nThe joke works by personifying the samosa and applying these culinary traits to mental health:\n\n1.  **"All crust on the outside":** In psychological terms, having a "tough exterior" or being "crusty" implies that a person projects an image of being strong, hardened, or even cold to protect themselves.\n2.  **"Totally mashed on the inside":** While

Updating State

In [74]:
#Changing the to topic to samoosa
new_config = workflow.update_state({"configurable" : {"thread_id" : 1, "checkpoint_id" : '1f1eafe6-d722-6438-8000-34fe4f27d3d1', 'checkpoint_ns' : ""}} ,{'topic' : 'samosa'})

In [77]:
workflow.get_state(new_config)

StateSnapshot(values={'topic': 'samosa'}, next=('joke_generator',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1eb01a-d8e6-6c68-8001-28f43a82fb52'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-11-28T07:59:46.448275+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1eafe6-d722-6438-8000-34fe4f27d3d1'}}, tasks=(PregelTask(id='bc21d0b1-ff97-0660-9cb8-9f87c514a2d3', name='joke_generator', path=('__pregel_pull', 'joke_generator'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [79]:
workflow.invoke(None, new_config)

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy?\n\nBecause it was **all crust on the outside, but totally mashed on the inside!**',
 'explanation': 'This joke is a clever play on words that uses the physical characteristics of a popular snack—the **samosa**—as a metaphor for human emotions.\n\nTo understand the humor, you first have to look at the literal structure of a samosa. It consists of a thick, deep-fried pastry shell that is firm and crunchy, commonly referred to as the **crust**. Inside that shell, the filling is almost always made of soft, spiced potatoes that have been completely **mashed**.\n\nThe joke works by personifying the samosa and applying these culinary traits to mental health:\n\n1.  **"All crust on the outside":** In psychological terms, having a "tough exterior" or being "crusty" implies that a person projects an image of being strong, hardened, or even cold to protect themselves.\n2.  **"Totally mashed on the inside":** While "mashed" literally 